# 01 · The agent loop from scratch

Build a working tool-using agent in about 30 lines of plain OpenAI SDK code: a tool registry, executing the model's tool calls, feeding results back as observations, a stop condition, a `max_steps` guard, errors returned to the model instead of crashing, and per-step token and cost accounting. Then run the same loop on the Responses API.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** [Function calling](../08_llm_apps/04_function_calling.ipynb)

## Why this matters in interviews

- "Build an agent" is the most common agentic coding round. Interviewers want the loop written from memory, not a framework call.
- The follow-ups are about the guards: step cap, cost ceiling, repeat detection, what happens when a tool fails, when the loop stops.
- Knowing that every step re-sends the whole history (so cost grows with every step) separates people who have run agents from people who have read about them.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag01` | What is an AI agent, and how is it different from a chain? |
| `ag02` | What is the core agent loop? |
| `ag03` | Are function calling and tool calling the same thing? |
| `ag16` | How do you stop an agent looping forever? |
| `ag19` | How do you handle a tool that fails? |
| `ag29` | What is the difference between an agent and a workflow, in production terms? |
| `ag34` | How do you decide the step budget for an agent? |
| `ag40` | What is the difference between an agent framework and rolling your own? |
| `ag42` | How do you keep an agent from doing something expensive by accident? |
| `fo32` | What is the difference between an LLM, an agent and a workflow? |

## 1. Agent vs chain vs workflow

**In plain English:** a *chain* (or *workflow*) is a recipe you wrote: step 1, step 2, step 3. An *agent* decides the steps itself: which tool to use, how many times, and when it is finished. The defining property is not tool use or memory. It is that **the next action depends on what the model just observed**.

An agent is four things:

| Part | What it is | Where it lives in the code below |
|---|---|---|
| **LLM** | decides the next action | `client.chat.completions.create(..., tools=TOOLS)` |
| **Tools** | functions the model may ask for | `REGISTRY` (name → Python function) and `TOOLS` (their JSON schemas) |
| **Loop** | call → execute → observe → repeat | `for step in range(max_steps)` |
| **State** | everything seen so far | the `messages` list |

| | Single call | Chain / workflow | Router | Agent |
|---|---|---|---|---|
| Who picks the next step | nobody | you, in code | model picks one branch | model, every step |
| LLM calls per request | 1 | fixed | 1 + the branch | variable, 1 … `max_steps` |
| Cost and latency | exact | predictable | predictable | unbounded without guards |
| How you test it | one input/output | node by node | router accuracy + each branch | whole trajectories |
| Use it when | one-shot task | the steps are known in advance | inputs fall into known kinds | the steps depend on what you find |

The engineering rule interviewers want to hear: **use the least agency that solves the problem**. Most production "agents" are workflows with one or two agentic steps.

In [ ]:
import json
import random
import time
import ast
import operator
import datetime as dt

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

random.seed(0)
rng = np.random.default_rng(0)


def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()


draw_pipeline(["user question", "retrieve docs", "LLM: answer", "format"], title="A chain: YOU fixed the steps")

In [ ]:
def draw_agent_loop():
    """The agent loop: the model decides, your code acts, the result goes back in."""
    fig, ax = plt.subplots(figsize=(10, 3.4))
    boxes = {"task": (0.0, 1.3, "user task"), "llm": (2.6, 1.3, "LLM decides\n(tool schemas offered)"),
             "run": (5.4, 2.5, "YOUR code runs\nthe tool"), "obs": (5.4, 0.1, "observation appended\nto messages"),
             "done": (8.2, 1.3, "final answer")}
    for x, y, label in boxes.values():
        ax.add_patch(FancyBboxPatch((x, y), 2.0, 0.9, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 1.0, y + 0.45, label, ha="center", va="center", fontsize=9)
    def arrow(a, b, text="", rad=0.0):
        ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->", lw=1.6, connectionstyle=f"arc3,rad={rad}"))
        if text:
            ax.text((a[0] + b[0]) / 2, (a[1] + b[1]) / 2 + 0.12, text, ha="center", fontsize=8, color="#c92a2a")
    arrow((2.05, 1.75), (2.55, 1.75))
    arrow((4.65, 2.0), (5.35, 2.8), "tool_calls")
    arrow((6.4, 2.45), (6.4, 1.05))
    arrow((5.35, 0.5), (4.1, 1.25), "loop", rad=-0.2)
    arrow((4.65, 1.75), (8.15, 1.75), "no tool calls (or a guard trips)")
    ax.set_xlim(-0.2, 10.4); ax.set_ylim(-0.1, 3.6); ax.axis("off")
    ax.set_title("An agent: the MODEL picks the next step, every step")
    plt.show()


draw_agent_loop()

The second picture is the whole idea. The model never executes anything: it *asks* for a tool call, and your code decides whether to honour it. That boundary is where permission checks, validation and approval gates live (notebooks [02](02_tool_design_and_function_schemas.ipynb) and [06](06_guardrails_hitl_and_failure_modes.ipynb)).

**Function calling vs tool calling (`ag03`):** the same mechanism. "Function calling" was OpenAI's 2023 name; "tools" is the current name, and it also covers provider-hosted tools, MCP servers and sub-agents.

## 2. Connect to a model

The first cell of every LLM notebook in this set. With `OPENAI_API_KEY` set it talks to OpenAI, with Ollama running it talks to a local model, otherwise an offline stand-in answers deterministically.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

**Offline note.** The stand-in is not intelligent. It picks the tool whose name and description best overlap your message, pulls arguments out of the message, and after tool results arrive it answers by quoting them ("Based on the tool results: …"). So offline, every run is *one round of tool calls, then an answer*. A real model can chain calls (use one result to decide the next call); the loop code is identical.

## 3. The tools: plain Python functions plus a registry

Three tools: a calculator, a search over a tiny knowledge base, and a clock. Two details matter:

- The calculator walks the syntax tree and allows only arithmetic. **Never `eval()` model output**: `eval("__import__('os').system('...')")` would run anything.
- The clock is frozen at 2026-09-29 12:00 UTC so the notebook prints the same thing every run.

In [ ]:
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg}


def calculator(expression: str) -> str:
    """Safe arithmetic: numbers and + - * / ** % only."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError(f"only arithmetic is allowed, got {type(node).__name__}")
    value = ev(ast.parse(expression, mode="eval").body)
    return str(int(value) if float(value).is_integer() else round(value, 6))


KB = {
    "refunds": "Refunds are issued within 14 days of purchase for unused items.",
    "shipping": "Standard shipping takes 3-5 business days; express shipping takes 1 business day.",
    "warranty": "The warranty covers manufacturing defects for 2 years.",
    "returns": "Items can be returned within 30 days in their original packaging.",
}
_STOP = set("a an the is are do does how what when for of to in on my i it and or can".split())


def search_kb(query: str) -> str:
    """Keyword search over KB; returns up to 2 passages tagged with their source."""
    q = {w.strip("?.,!").lower() for w in query.split()} - _STOP
    scored = []
    for doc_id, text in KB.items():
        words = {w.strip("?.,;!").lower() for w in text.split()} | {doc_id, doc_id.rstrip("s")}
        overlap = len(q & words)
        if overlap:
            scored.append((overlap, doc_id, text))
    if not scored:
        return f"No passages match {query!r}. Topics available: {', '.join(KB)}."
    scored.sort(key=lambda t: -t[0])
    return "\n".join(f"[{d}] {t}" for _, d, t in scored[:2])


UTC_OFFSET = {"London": 1, "Tokyo": 9, "New York": -4}
FROZEN_NOW = dt.datetime(2026, 9, 29, 12, 0, tzinfo=dt.timezone.utc)


def get_time(city: str) -> str:
    if city not in UTC_OFFSET:
        raise KeyError(f"unknown city {city!r}; known cities: {', '.join(UTC_OFFSET)}")
    local = FROZEN_NOW + dt.timedelta(hours=UTC_OFFSET[city])
    return f"{city}: {local:%H:%M} local time"


REGISTRY = {f.__name__: f for f in (calculator, search_kb, get_time)}

print("calculator('17*23')        ->", calculator("17*23"))
print("search_kb('express ship')  ->", search_kb("How long does express shipping take?"))
print("get_time('Tokyo')          ->", get_time("Tokyo"))
try:
    calculator("__import__('os').getcwd()")
except ValueError as e:
    print("calculator(<code>)         -> refused:", e)
assert calculator("17*23") == "391" and calculator("(2+3)/4") == "1.25"

The model never sees the Python functions, only their **schemas**: a name, a description (which *is* a prompt: it is how the model decides) and a JSON Schema for the arguments. Notebook [02](02_tool_design_and_function_schemas.ipynb) is about writing these well and generating them from type hints.

In [ ]:
def tool_schema(name, description, properties):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties,
                       "required": list(properties), "additionalProperties": False}}}


TOOLS = [
    tool_schema("calculator", "Evaluate an arithmetic expression such as 17*23 or (2+3)/4.",
                {"expression": {"type": "string", "description": "Arithmetic only, e.g. '12 * 7'"}}),
    tool_schema("search_kb", "Search the company knowledge base for policies: refunds, shipping, warranty, returns.",
                {"query": {"type": "string"}}),
    tool_schema("get_time", "Get the current local time in a city.",
                {"city": {"type": "string", "description": "City name, e.g. 'Tokyo'"}}),
]
print(json.dumps(TOOLS[2], indent=2))

## 4. The loop in about 30 lines

Read it top to bottom; it is the answer to `ag02`.

1. Call the model with the conversation and the tool schemas.
2. If the reply has **no tool calls**, the model has answered: stop.
3. Otherwise append the assistant message (with its `tool_calls`), run each call **in your code**, and append each result as a `tool` message carrying the matching `tool_call_id`.
4. Repeat, but never more than `max_steps` times.

`execute()` turns every failure (unknown tool, bad JSON, an exception inside the tool) into a string observation. That is deliberate: see section 5.

In [ ]:
SYSTEM = "You are a support assistant. Use the tools when they help, then answer briefly."


def execute(call, registry=REGISTRY) -> str:
    """Run one tool call. Failures become observations the model can read, never exceptions."""
    name = call.function.name
    if name not in registry:
        return f"ERROR: there is no tool named {name!r}. Available tools: {', '.join(registry)}."
    try:
        args = json.loads(call.function.arguments or "{}")
        return str(registry[name](**args))
    except Exception as e:
        return f"ERROR {type(e).__name__}: {e}"


def run_agent(task, max_steps=6, tools=TOOLS, registry=REGISTRY):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": task}]
    trace = []                                           # one row per LLM call
    for step in range(1, max_steps + 1):
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        msg = resp.choices[0].message
        trace.append({"step": step, "prompt_tokens": resp.usage.prompt_tokens,
                      "completion_tokens": resp.usage.completion_tokens, "tool_calls": len(msg.tool_calls or [])})
        if not msg.tool_calls:                           # stop condition: the model answered
            messages.append({"role": "assistant", "content": msg.content})
            return {"answer": msg.content, "status": "completed", "messages": messages, "trace": trace}
        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for call in msg.tool_calls:                      # the model asked; OUR code acts
            messages.append({"role": "tool", "tool_call_id": call.id, "content": execute(call, registry)})
    return {"answer": None, "status": "max_steps", "messages": messages, "trace": trace}

In [ ]:
def show_transcript(messages, width=100):
    for m in messages:
        role = m["role"]
        if role == "assistant" and m.get("tool_calls"):
            for tc in m["tool_calls"]:
                print(f"[assistant] -> {tc['function']['name']}({tc['function']['arguments']})")
        else:
            text = str(m.get("content")).replace("\n", " | ")
            print(f"[{role}]".ljust(12), text[:width] + ("..." if len(text) > width else ""))


result = run_agent("What is 12 * 7, and what time is it in Tokyo?")
show_transcript(result["messages"])
print("\nstatus:", result["status"], "| LLM calls:", len(result["trace"]))

What happened:

- **Step 1**: the model returned two tool calls *in one message* (parallel tool calls) and no text. Our code ran both and appended two `tool` messages.
- **Step 2**: the model saw the observations and answered without calling a tool, so the loop stopped.

A real model would phrase the answer ("12 × 7 = 84, and it is 21:00 in Tokyo"); the stand-in quotes the tool results. Pass `parallel_tool_calls=False` to force one call per step.

The API enforces the bookkeeping: every assistant `tool_calls` entry must be answered by a `tool` message with the same `tool_call_id`, before the next assistant turn.

In [ ]:
msgs = result["messages"]
call_ids = [tc["id"] for m in msgs if m["role"] == "assistant" for tc in m.get("tool_calls") or []]
tool_ids = [m["tool_call_id"] for m in msgs if m["role"] == "tool"]
assert result["status"] == "completed" and len(result["trace"]) == 2
assert call_ids == tool_ids and len(tool_ids) == 2, "every tool call must be answered, in order"

import openai
broken = msgs[:3]                     # system, user, assistant-with-tool_calls ... but no tool messages
broken = broken + [{"role": "user", "content": "and?"}]
try:
    client.chat.completions.create(model=MODEL, messages=broken, tools=TOOLS)
except openai.BadRequestError as e:
    print("Forgot the tool messages -> 400:", str(e)[:150], "...")

## 5. Errors are observations, not exceptions

**In plain English:** when a tool fails, tell the model *what* went wrong in a sentence it can act on. Do not crash the run, and do not hand it a stack trace.

`get_time("Atlantis")` raises `KeyError`. Watch what the model receives.

In [ ]:
result = run_agent("What time is it in Atlantis?")
show_transcript(result["messages"])
tool_obs = [m["content"] for m in result["messages"] if m["role"] == "tool"][0]
assert tool_obs.startswith("ERROR") and "known cities" in tool_obs
assert result["status"] == "completed"

The observation names the mistake *and the legal values* ("known cities: London, Tokyo, New York"). A real model then either asks the user which city they meant or retries with a valid one; with a bare `KeyError: 'Atlantis'` it has nothing to go on. Two rules for `ag19`:

| Failure | What to do |
|---|---|
| Wrong arguments (a 4xx, a validation error) | Return an instructive message. An identical retry is pure waste. |
| Transient (timeout, 5xx, 429) | Retry once or twice with backoff *inside the tool*, then report failure. |
| The same tool keeps failing | Stop offering it for the rest of the run and say it is unavailable. |

And an unknown tool name (the model invented one) is also just an observation:

In [ ]:
class FakeCall:                       # the same shape as an SDK tool call, for a quick test
    def __init__(self, name, arguments):
        self.id, self.function = "call_test", type("F", (), {"name": name, "arguments": arguments})()


print(execute(FakeCall("get_weather", '{"city": "Paris"}')))
print(execute(FakeCall("calculator", '{"expression": "10 / 0"}')))
print(execute(FakeCall("calculator", '{"expr": "1+1"}')))

## 6. Guards: why `max_steps` is not optional

Here is a realistic bug. A "history trimmer" meant to save tokens keeps only the system and user messages, so **tool results never reach the model**. The model keeps asking for the same thing, and the only thing that stops the run is `max_steps`.

In [ ]:
def amnesiac_create(messages, tools):
    """BUG: drops assistant and tool messages before calling the model."""
    kept = [m for m in messages if m["role"] in ("system", "user")]
    return client.chat.completions.create(model=MODEL, messages=kept, tools=tools)


def run_buggy(task, max_steps=6):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": task}]
    calls = []
    for step in range(1, max_steps + 1):
        msg = amnesiac_create(messages, TOOLS).choices[0].message
        if not msg.tool_calls:
            return "completed", calls
        messages.append({"role": "assistant", "content": None, "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for call in msg.tool_calls:
            calls.append((call.function.name, call.function.arguments))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": execute(call)})
    return "max_steps", calls


status, calls = run_buggy("What time is it in Tokyo?")
print("status:", status)
for i, c in enumerate(calls, 1):
    print(f"  step {i}: {c[0]}({c[1]})")
assert status == "max_steps" and len(set(calls)) == 1

Six identical calls. With a real model and a paid tool this is the "$14 request" story from `ag42`. The production loop adds cheap guards that catch it early:

| Guard | Catches | Typical setting |
|---|---|---|
| step cap (`max_steps`) | any loop | 8-15 for tool-using agents; set from the p95 of *successful* runs (`ag34`) |
| repeat-call detector | same tool + same arguments again | tell the model once, stop on the second repeat |
| token / money budget | a few huge steps | checked **before** each call, not after |
| wall-clock deadline | slow tools | shorter than user patience |
| give-up path | all of the above | return what was found and what was tried, never a bare error |

Below, the same loop with the repeat detector and a token budget, and a named termination reason for every exit. The model call is a parameter (`create`), so we can inject the buggy trimmer and show the detector stopping it.

In [ ]:
PRICE_PER_M = {"input": 0.40, "output": 1.60}    # USD per 1M tokens, gpt-4.1-mini list price when written


def cost_usd(prompt_tokens, completion_tokens):
    return (prompt_tokens * PRICE_PER_M["input"] + completion_tokens * PRICE_PER_M["output"]) / 1e6


def default_create(messages, tools):
    return client.chat.completions.create(model=MODEL, messages=messages, tools=tools)


def run_agent_guarded(task, messages=None, create=default_create, max_steps=8, max_repeats=1, max_tokens=20_000):
    messages = messages if messages is not None else [{"role": "system", "content": SYSTEM}]
    messages.append({"role": "user", "content": task})
    trace, seen, repeats, used = [], {}, 0, 0
    for step in range(1, max_steps + 1):
        est_next = len(json.dumps(messages)) // 4                      # rough prompt size, before calling
        if used + est_next > max_tokens:
            return finish("token_budget", None, messages, trace, seen)
        resp = create(messages, TOOLS)
        msg, u = resp.choices[0].message, resp.usage
        used += u.total_tokens
        trace.append({"step": step, "prompt_tokens": u.prompt_tokens, "completion_tokens": u.completion_tokens,
                      "usd": cost_usd(u.prompt_tokens, u.completion_tokens)})
        if not msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content})
            return finish("completed", msg.content, messages, trace, seen)
        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for call in msg.tool_calls:
            key = (call.function.name, call.function.arguments)
            if key in seen:
                repeats += 1
                obs = f"You already called {key[0]} with these arguments; the result was: {seen[key]}. Do not repeat it."
            else:
                obs = seen[key] = execute(call)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": obs})
        if repeats > max_repeats:
            return finish("stuck_repeating", None, messages, trace, seen)
    return finish("max_steps", None, messages, trace, seen)


def finish(status, answer, messages, trace, seen):
    if answer is None:                                                  # the give-up path: say what we have
        found = "; ".join(f"{name} -> {obs[:60]}" for (name, _), obs in seen.items()) or "nothing"
        answer = f"I could not finish ({status}). What I found so far: {found}"
    return {"answer": answer, "status": status, "messages": messages, "trace": trace}

In [ ]:
buggy = run_agent_guarded("What time is it in Tokyo?", create=amnesiac_create)
print("status:", buggy["status"], "after", len(buggy["trace"]), "LLM calls")
print("answer:", buggy["answer"])
assert buggy["status"] == "stuck_repeating" and len(buggy["trace"]) < 6

ok = run_agent_guarded("What is 12 * 7, and what time is it in Tokyo?")
print("\nhealthy run:", ok["status"], "|", ok["answer"])
assert ok["status"] == "completed"

The detector stopped the broken run after 3 calls instead of 6, and the user got a partial answer that says what happened. Log the termination reason of every run: if a large share end at `max_steps` or `stuck_repeating`, the agent is not succeeding, it is running out of budget (`ag34`). Raising the cap is almost never the fix.

## 7. Per-step token and cost accounting

**In plain English:** the model has no memory between calls. Every step sends the *whole* conversation again: system prompt, tool schemas, every earlier tool result. So an agent's cost per step grows as it works.

Below, one conversation of four user turns. Each turn is a tool round-trip (two LLM calls), and the history keeps growing.

In [ ]:
session = None
turns = ["How long does express shipping take?", "And what is 12 * 7?",
         "What time is it in London?", "What is the refund policy?"]
all_steps = []
for t in turns:
    r = run_agent_guarded(t, messages=session)
    session = r["messages"]
    for row in r["trace"]:
        all_steps.append({**row, "turn": t})
    print(f"{t:40s} -> {r['answer'][:70]}")

prompt = np.array([s["prompt_tokens"] for s in all_steps])
completion = np.array([s["completion_tokens"] for s in all_steps])
usd = np.array([s["usd"] for s in all_steps])
print(f"\nLLM calls: {len(all_steps)} | prompt tokens per call: {prompt.tolist()}")
print(f"total: {prompt.sum()} prompt + {completion.sum()} completion tokens = ${usd.sum():.6f}")
assert np.all(np.diff(prompt) >= 0), "each call re-sends the growing history"

In [ ]:
x = np.arange(1, len(all_steps) + 1)
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(x, prompt, label="prompt (input) tokens", color="#4c6ef5")
ax.bar(x, completion, bottom=prompt, label="completion (output) tokens", color="#fab005")
ax.set_xlabel("LLM call number in the session"); ax.set_ylabel("tokens")
ax.set_title("Every call re-sends the whole history, so input tokens grow")
ax2 = ax.twinx()
ax2.plot(x, np.cumsum(usd) * 1e6, "o-", color="#c92a2a", label="cumulative cost")
ax2.set_ylabel("cumulative cost (micro-USD)")
ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="lower right", fontsize=8)
plt.show()

Offline, the token counts come from the stand-in's rough "characters / 4" estimate, and the absolute dollars are tiny because the conversation is short. The *shape* is the lesson, and it is the same on a real model: input tokens dominate, they grow with every step, and with real tool schemas (hundreds of tokens each) and real tool outputs (often thousands) the growth is much steeper.

Three consequences:

- **Tool outputs are the cost driver.** Shape them before they enter the history (notebook [02](02_tool_design_and_function_schemas.ipynb)).
- **Keep the prefix stable** (system prompt and tools in a fixed order) so provider prompt caching can discount the repeated part.
- **Alarm on cost per request (p99), not on monthly spend** (`ag42`).

## 8. The same loop on the Responses API

OpenAI's newer Responses API has the same loop with different nouns:

| | Chat Completions | Responses |
|---|---|---|
| conversation | `messages=[...]` | `input=[...]` (items) |
| tool schema | `{"type": "function", "function": {name, parameters}}` | flat: `{"type": "function", "name", "parameters"}` |
| model asks for a tool | `message.tool_calls[i]` | output item with `type == "function_call"` |
| you return the result | `{"role": "tool", "tool_call_id", "content"}` | `{"type": "function_call_output", "call_id", "output"}` |
| server-side history | no | optional: `previous_response_id` |

Chat Completions stays the default in these notebooks because Ollama, vLLM, LiteLLM and most gateways speak it.

In [ ]:
RESP_TOOLS = [{"type": "function", "name": t["function"]["name"], "description": t["function"]["description"],
               "parameters": t["function"]["parameters"], "strict": True} for t in TOOLS]


def run_agent_responses(task, max_steps=6):
    items = [{"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        resp = client.responses.create(model=MODEL, instructions=SYSTEM, input=items, tools=RESP_TOOLS)
        calls = [o for o in resp.output if o.type == "function_call"]
        if not calls:
            return resp.output_text, step
        items += resp.output                                   # keep the model's function_call items
        for c in calls:
            call = FakeCall(c.name, c.arguments)
            items.append({"type": "function_call_output", "call_id": c.call_id, "output": execute(call)})
    return None, max_steps


answer, steps = run_agent_responses("What is 12 * 7, and what time is it in Tokyo?")
print(f"Responses API, {steps} calls ->", answer)
chat_answer = run_agent("What is 12 * 7, and what time is it in Tokyo?")["answer"]
assert answer == chat_answer, "same loop, same tools, same answer"

`strict: True` on each tool makes the model's arguments match the schema exactly (it requires `additionalProperties: false` and every property in `required`; notebook [02](02_tool_design_and_function_schemas.ipynb) covers it).

## Try it yourself

**Exercise 1.** Add a `convert_usd(amount, to_currency)` tool with fixed rates (EUR 0.9, GBP 0.8, JPY 150) and an `enum` for `to_currency`. Run "Convert 100 USD to EUR" through `run_agent`, passing your own tools and registry.

In [ ]:
# Solution — try it yourself first, then run this
RATES = {"EUR": 0.9, "GBP": 0.8, "JPY": 150}


def convert_usd(amount: float, to_currency: str) -> str:
    if to_currency not in RATES:
        raise ValueError(f"to_currency must be one of {list(RATES)}")
    return f"{amount} USD = {amount * RATES[to_currency]:.2f} {to_currency}"


my_tools = TOOLS + [tool_schema("convert_usd", "Convert an amount of US dollars (USD) into another currency.",
                                {"amount": {"type": "number"},
                                 "to_currency": {"type": "string", "enum": list(RATES)}})]
my_registry = {**REGISTRY, "convert_usd": convert_usd}
r = run_agent("Convert 100 USD to EUR", tools=my_tools, registry=my_registry)
show_transcript(r["messages"][1:])
assert "90.00 EUR" in r["answer"]

**Exercise 2.** Add a wall-clock deadline to the guarded loop: stop with status `"deadline"` if more than `deadline_s` seconds have passed before the next model call. Test it with a slow model wrapper that sleeps 0.2 s per call: the Tokyo question needs two calls, so `deadline_s=0.1` should trip and `deadline_s=5` should not.

In [ ]:
# Solution — try it yourself first, then run this
def slow_create(messages, tools):
    time.sleep(0.2)
    return default_create(messages, tools)


def with_deadline(create, deadline_s):
    start = time.perf_counter()
    def wrapped(messages, tools):
        if time.perf_counter() - start > deadline_s:
            raise TimeoutError("deadline")
        return create(messages, tools)
    return wrapped


def run_with_deadline(task, deadline_s):
    try:
        return run_agent_guarded(task, create=with_deadline(slow_create, deadline_s))["status"]
    except TimeoutError:
        return "deadline"


tight = run_with_deadline("What time is it in Tokyo?", 0.1)
loose = run_with_deadline("What time is it in Tokyo?", 5.0)
print("deadline 0.1 s:", tight)
print("deadline 5 s  :", loose)
assert (tight, loose) == ("deadline", "completed")

**Exercise 3.** Termination-reason dashboard: run the guarded agent on the five tasks below (one with the buggy trimmer) and count the termination reasons. What share "succeeded"?

In [ ]:
# Solution — try it yourself first, then run this
from collections import Counter

jobs = [("What is 3 * 3?", default_create), ("What time is it in Tokyo?", default_create),
        ("What is the warranty period?", default_create), ("What time is it in London?", amnesiac_create),
        ("How long does standard shipping take?", default_create)]
reasons = Counter(run_agent_guarded(t, create=c)["status"] for t, c in jobs)
print(dict(reasons), f"-> success rate {reasons['completed'] / len(jobs):.0%}")

## Say it in an interview

**30-second answer (`ag02`).** "An agent is an LLM in a loop with tools. I send the conversation plus tool schemas; if the model returns tool calls, *my code* validates and executes them and appends each result as a tool message with the matching id; I repeat until the model answers without a tool call or a guard trips. The model never executes anything, which is why permissions and approvals live in my code. The guards are a step cap, a token and money budget checked before each call, a repeat-call detector, a deadline, and an explicit give-up path that returns what was found."

**Agent vs workflow (`ag01`, `ag29`, `fo32`).** A workflow is a fixed graph you wrote; an agent lets the model choose the next step from what it just observed. Agents buy flexibility with unpredictable cost, latency and testing. Start with the least agency that works.

**Numbers and trade-offs.**
- Step caps of 8-15 cover most tool-using tasks; set the cap from the p95 of successful runs and watch the termination-reason distribution.
- Cost per step grows because the whole history is re-sent every call; tool outputs are usually the biggest term.
- Framework vs hand-rolled (`ag40`): the loop is ~30 lines, so hand-roll it to understand and control it; adopt LangGraph when you need checkpointing, durable interrupts or a complex graph.

**Traps they probe.**
- Forgetting to append tool results (the API returns 400, or a trimmer silently drops them and the agent loops).
- Letting a tool exception crash the run instead of returning it as an observation.
- `while True` with no step cap; checking the budget *after* the call.
- Treating "no tool call" as proof of success; it is the weakest stop signal ("the agent that does nothing", `ag38`).
- `eval()` in a calculator tool.

## Next

- [02 · Tool design and function schemas](02_tool_design_and_function_schemas.ipynb): descriptions, strict schemas, observation shaping, security.
- [03 · ReAct and plan-and-execute](03_react_and_plan_and_execute.ipynb): text ReAct, planners and reflection.
- [06 · Guardrails, HITL and failure modes](06_guardrails_hitl_and_failure_modes.ipynb): budgets and the failure playbook in depth.
- The same loop in frameworks: [LangChain agents](../13_langchain/06_agents_and_memory.ipynb), [LangGraph ReAct agent](../14_langgraph/05_react_tool_calling_agent.ipynb).
- Coding round practice: [Build an agent in 30 minutes](../17_interview_coding_rounds/02_build_an_agent_in_30_minutes.ipynb).
- Theory: [agentic_ai_flow course](../../agentic_ai_flow/index.html) (chapters *The agent loop*, *Naive loops*) and the [interview bank](../../ai_interview_prep/index.html).